## What the Inputs of the Queueing Formula Cost

**Question.** In the base case of the evaluation (`2_evaluation.ipynb`, B.1), the Kingman baseline is the tightest method
that uses execution times only, and it does not hold: its coverage falls below the confidence level. Is that because
its queueing formula is wrong, or because the formula is given a wrong mean or variance of the execution time? This
notebook answers that on the chain, the held-out configurations and the per-service models of B.1. It is a diagnostic
and not a further baseline: three of its five variants use the true execution times, which no method has.

The notebook is structured as follows:

1. **Setup:** the data and the models of B.1.
2. **The formula and its five variants:** the same formula with predicted, true and pessimistic inputs.
3. **Results:** coverage and tightness of every variant, and where the baseline misses.

### 1. Setup

The cell below loads the chain of B.1 and fits the per-service models of the execution time for every seed, with the
parameters of `2_evaluation.ipynb`. They must be kept equal to the ones there, otherwise the two notebooks describe
different models:

- `data_source`: "synthetic" (generated services and a simulated chain of queues) or "real" (measured processing times
  of the testbed campaign, replayed through the same queues).
- `alpha`: the violation probability $\varepsilon$; a method at `alpha` gives the $(1-\varepsilon)$ quantile of the
  end-to-end latency (s).
- `confidence`: the probability with which the pessimistic mean of Pegasus lies above the true mean, and the coverage
  that a method must reach under every seed to *hold*.
- `seeds`: every seed determines the training data and the randomness of the ground truth (with synthetic data also
  the generated service types).

*Coverage* of a variant under a seed is the share of the held-out configurations at which its latency is not below the
lower edge of the confidence interval of the measured quantile (`coverage_check`). *Tightness* at a configuration is
the latency of the variant ÷ the measured quantile: 1 is exact, above 1 is looser.

In [1]:
# Data source, as in 2_evaluation.ipynb:
# - "synthetic": generated services and a simulated chain of queues (pegasus/data.py, pegasus/simulation.py)
# - "real": measured processing times of the testbed campaign in statics/results_milos, replayed through the same queues (pegasus/real_data.py)
data_source = "real"

import warnings

import numpy as np
from IPython.display import Markdown, display
from sklearn.exceptions import ConvergenceWarning

from pegasus.data import assert_unseen, load_chain_data, synthetic_chain, synthetic_execution_moments
from pegasus.evaluation import arrival_text, coverage, fit_services, pegasus_bound, queueing_formula_inputs
from pegasus.real_data import chain_execution_moments

# The parameters of 2_evaluation.ipynb (first code cell); keep them equal to the ones there
alpha = 0.05                                           # violation probability epsilon: a method at alpha gives the (1 - alpha) latency quantile
confidence = 0.95                                      # the pessimistic mean of a bound lies above the true mean with this probability
coverage_check = dict(n_batches=10, confidence=0.95)   # confidence interval of a measured quantile (batch means)
seed = 35                                              # base seed of the data source
seeds = list(range(10))                                # the diagnostic is repeated for these seeds
p_label = f"p{100 * (1 - alpha):g}"

if data_source == "synthetic":
    chain_options = lambda s: dict(path=synthetic_chain(3), service_seed=s)               # the chain of B.1 under the seed s
    # true mean and variance of the execution time (s, s^2) per held-out configuration and service: the parameters of the generated services
    true_moments = lambda s: synthetic_execution_moments(synthetic_chain(3), bases[s].x_test, service_seed=s)
else:
    base_experiment = "04e-chain-qr-cv-face"                                              # the chain of B.1
    base_utilization = 0.5                                                                # utilization of the busiest service of every run in B.1
    chain_options = lambda s: dict(experiment=base_experiment, utilization=base_utilization)
    # true mean and variance: those of the processing times that the testbed measured in every run of the chain; the same under every seed
    true_moments = lambda s: chain_execution_moments(base_experiment)

# Data of the chain and the per-service GPs for every seed; no model sees a run of the evaluated chain. The DNNs of B.1 are
# not needed: the formula uses the GPs only.
bases, base_models = {}, {}
with warnings.catch_warnings():
    # for some services the fitted GP hyperparameters end at their bounds, and scikit-learn warns for every such model
    warnings.simplefilter("ignore", ConvergenceWarning)
    for s in seeds:
        bases[s] = load_chain_data(data_source, level=1 - alpha, coverage_check=coverage_check, seed=[seed, s], **chain_options(s))
        assert_unseen(bases[s])
        base_models[s] = fit_services(bases[s], bases[s].x_train, bases[s].y_train, model_seed=seed + s, confidence=confidence, with_dnn=False)
base = bases[seeds[0]]   # for what is the same under every seed: the names in the chain and the workload
arrivals = arrival_text(base.workload)
print(f"{data_source} data, {len(seeds)} seeds: chain {' -> '.join(base.path)}, {len(base.measured)} held-out configurations per seed, {arrivals}")

real data, 10 seeds: chain qr -> cv -> face, 48 held-out configurations per seed, poisson arrivals at one rate per configuration (3.48 to 42.2/s)


### 2. The Formula and Its Five Variants

**The formula.** For one service, the Kingman baseline computes the $(1-\varepsilon)$ latency quantile from two numbers,
the mean $m$ (s) and the variance (s²) of the execution time of one item, and from the arrival rate $\lambda$
(items/s). The mean waiting time is $\rho / (1 - \rho)$ × a factor for the variability of arrivals and execution times
× $m$, with the utilization $\rho = \lambda m$ (Kingman's approximation); the waiting time is then assumed to be
exponentially distributed. The quantiles of the services of the chain are added (`queueing_formula_latency` in
`pegasus/baselines.py`).

**Why the mean matters most.** The factor $\rho / (1 - \rho)$ grows faster and faster as $\rho$ approaches 1, so an
error in $m$ changes the waiting time by much more than its own size. Example: with $\lambda$ = 25 items/s and a true
mean of 20 ms, $\rho$ = 0.5 and the factor is 0.5 / 0.5 = 1. With a predicted mean of 14 ms, 30% too low, $\rho$ = 0.35
and the factor is 0.35 / 0.65 ≈ 0.54. The waiting time scales with the factor × the mean, 1 × 20 ms against
0.54 × 14 ms ≈ 7.5 ms: a mean that is 30% too low gives a waiting time that is almost three times too small.

**The five variants.** Each is the same formula with other inputs (`queueing_formula_inputs` in
`pegasus/evaluation.py`):
- *predicted mean, predicted variance:* the mean and the variance that the GP of a service predicts. This is the
  baseline of B.1.
- *predicted mean, true variance* and *true mean, predicted variance:* one of the two inputs replaced by its true
  value, which shows which input the misses come from.
- *true mean, true variance:* the formula with correct inputs. What it misses is the error of the formula itself.
- *pessimistic mean, predicted variance:* the mean that Pegasus uses, i.e. the prediction shifted up by the GP's
  uncertainty about the mean, so that it lies above the true mean with probability `confidence`.

The *true* mean and variance are those of the execution times that the ground truth draws from: with synthetic data
the parameters of the generated service, with real data the mean and the variance of the processing times that the
testbed measured in the run (`true_moments` in the setup cell).

**None of the variants is a guarantee,** also not the one with the pessimistic mean, for three reasons. The exponential
waiting time is an approximation and not an upper bound. The quantiles of the services are added at the full
$\varepsilon$ per service, although the quantile of a sum can exceed the sum of the quantiles; a valid sum needs
$\varepsilon / K$ per service for a chain of $K$ services. And every service is treated as if it received the external
arrivals. Pegasus per node repairs all three: it uses Kingman's *bound* $P(\text{wait} > w) \le e^{-\theta w}$ instead
of Kingman's *approximation* of the mean wait, splits $\varepsilon$ over the services, and models the arrivals of the
later services. Its bound is computed in the cell below as the reference for the tightness.

In [2]:
# The queueing formula of the Kingman baseline with predicted, true and pessimistic inputs, and the bound of Pegasus per node as reference
formula_variants, mean_error, pegasus_latency = {}, {}, {}   # seed -> variant -> latency per held-out configuration (s) / relative error of the
                                                             # predicted mean at the busiest service / latency of Pegasus per node (s)
for s in seeds:
    formula_variants[s], mean_error[s] = queueing_formula_inputs(bases[s], base_models[s], *true_moments(s), alpha)
    pegasus_latency[s] = pegasus_bound(bases[s], base_models[s], alpha)
print(f"{len(formula_variants[seeds[0]])} variants computed for {len(seeds)} seeds.")

5 variants computed for 10 seeds.


### 3. Results

**Table.** One row per variant. *Coverage* is given per seed, one value per seed in the order of `seeds`. A variant
*holds* if its coverage reaches `confidence` under every seed, the rule of B.1: the worst seed decides. *Median
tightness* is the median over the held-out configurations of all seeds of the variant's latency ÷ the measured
quantile. The conclusion below the table is rendered from the same results. Read the first four rows as a 2 × 2
comparison: going from a predicted to a true variance (row 1 to 2, row 3 to 4) and from a predicted to a true mean
(row 1 to 3, row 2 to 4).

In [3]:
variant_coverage = {name: [coverage(formula_variants[s][name], bases[s].measured_low) for s in seeds] for name in formula_variants[seeds[0]]}
variant_tightness = {name: np.median(np.concatenate([formula_variants[s][name] / bases[s].measured for s in seeds])) for name in variant_coverage}
pegasus_tightness = np.median(np.concatenate([pegasus_latency[s] / bases[s].measured for s in seeds]))
baseline_variant = next(iter(variant_coverage))   # predicted mean, predicted variance
errors = np.concatenate([mean_error[s] for s in seeds])
covered = np.concatenate([formula_variants[s][baseline_variant] >= bases[s].measured_low for s in seeds])   # the baseline, per held-out configuration
too_low = errors < 0                                                                                        # the GP underestimates the mean of the busiest service
display(Markdown(
    "| Inputs of the formula | Coverage per seed | Holds | Median tightness |\n|---|---|---|---|\n"
    + "\n".join(f"| {name} | {', '.join(f'{c:.2f}' for c in variant_coverage[name])} | {'yes' if min(variant_coverage[name]) >= confidence else 'no'} "
                f"| {variant_tightness[name]:.2f}× |" for name in variant_coverage)
    + f"\n\n**What this shows** ({data_source} data, {arrivals}, ε = {alpha:g}, {len(seeds)} seeds).\n"
    f"- **Error of the predicted mean.** At the busiest service of a held-out configuration, the mean that the GP predicts differs from the true mean "
    f"by {np.median(np.abs(errors)):.0%} in the median and by {np.quantile(np.abs(errors), 0.9):.0%} at the 90th percentile of the configurations "
    f"(all seeds). The GP predicts too low a mean at {np.mean(too_low):.0%} of them.\n"
    f"- **Where the baseline misses.** Its coverage is "
    + (f"{np.mean(covered[too_low]):.0%}" if too_low.any() else "not defined")
    + " at the configurations with too low a predicted mean and "
    + (f"{np.mean(covered[~too_low]):.0%}" if (~too_low).any() else "not defined")
    + " at the others.\n"
    f"- **Which input matters.** Compare the first four rows of the table: replacing the predicted variance by the true one changes the lowest "
    f"coverage over the seeds from {min(variant_coverage[baseline_variant]):.2f} to {min(variant_coverage['predicted mean, true variance']):.2f}, "
    f"replacing the predicted mean by the true one changes it to {min(variant_coverage['true mean, predicted variance']):.2f}, and with both true "
    f"inputs it is {min(variant_coverage['true mean, true variance']):.2f}.\n"
    f"- **What a margin for the error of the mean costs.** With the pessimistic mean the lowest coverage over the seeds is "
    f"{min(variant_coverage['pessimistic mean, predicted variance']):.2f} and the median tightness "
    f"{variant_tightness['pessimistic mean, predicted variance']:.2f}×, against {variant_tightness[baseline_variant]:.2f}× for the baseline and "
    f"{pegasus_tightness:.2f}× for Pegasus per node ({p_label} of the end-to-end latency)."
))

| Inputs of the formula | Coverage per seed | Holds | Median tightness |
|---|---|---|---|
| predicted mean, predicted variance (the baseline) | 0.94, 0.85, 0.85, 0.90, 0.81, 0.92, 0.92, 0.94, 0.88, 0.92 | no | 1.25× |
| predicted mean, true variance | 0.98, 0.96, 0.92, 0.94, 0.88, 0.96, 0.96, 0.96, 0.98, 0.94 | no | 1.25× |
| true mean, predicted variance | 0.98, 0.96, 1.00, 0.98, 0.98, 1.00, 1.00, 0.98, 1.00, 1.00 | yes | 1.21× |
| true mean, true variance | 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00 | yes | 1.20× |
| pessimistic mean, predicted variance | 1.00, 1.00, 0.98, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00 | yes | 1.93× |

**What this shows** (real data, poisson arrivals at one rate per configuration (3.48 to 42.2/s), ε = 0.05, 10 seeds).
- **Error of the predicted mean.** At the busiest service of a held-out configuration, the mean that the GP predicts differs from the true mean by 6% in the median and by 20% at the 90th percentile of the configurations (all seeds). The GP predicts too low a mean at 47% of them.
- **Where the baseline misses.** Its coverage is 77% at the configurations with too low a predicted mean and 100% at the others.
- **Which input matters.** Compare the first four rows of the table: replacing the predicted variance by the true one changes the lowest coverage over the seeds from 0.81 to 0.88, replacing the predicted mean by the true one changes it to 0.96, and with both true inputs it is 1.00.
- **What a margin for the error of the mean costs.** With the pessimistic mean the lowest coverage over the seeds is 0.98 and the median tightness 1.93×, against 1.25× for the baseline and 2.48× for Pegasus per node (p95 of the end-to-end latency).